# Vertex AI + Colab: GPTQ vs AWQ on Phi-3-mini → GGUF
This notebook sets up the environment, runs quantization/evaluation on CPU (or GPU if present), and optionally stores artifacts in GCS or runs on a Vertex AI VM.


In [ ]:
# !pip -q install -r requirements.txt
import sys, os, subprocess, json, psutil, platform
print('Python', sys.version); print('RAM (GB):', psutil.virtual_memory().total/1e9); print('System:', platform.platform())


## (Optional) Authenticate to Google & set GCS bucket
Uncomment the cell below to enable saving artifacts to your bucket.

In [ ]:
# from google.colab import auth
# auth.authenticate_user()
# PROJECT_ID = 'YOUR_PROJECT_ID'
# BUCKET = 'gs://YOUR_BUCKET'
# !gcloud config set project $PROJECT_ID


## Quantize: GPTQ (4-bit)

In [ ]:
!python /content/tinyllm-phase2/scripts/quantize_gptq.py --model microsoft/Phi-3-mini-4k-instruct --out /content/tinyllm-phase2/models/phi3mini-gptq-4bit --bits 4 --group_size 128


## Quantize: AWQ (4-bit)

In [ ]:
!python /content/tinyllm-phase2/scripts/quantize_awq.py --model microsoft/Phi-3-mini-4k-instruct --out /content/tinyllm-phase2/models/phi3mini-awq-4bit --w_bits 4 --group_size 128


## Evaluate (FP16, GPTQ, AWQ)

In [ ]:
!python /content/tinyllm-phase2/scripts/evaluate.py --which fp16 --model microsoft/Phi-3-mini-4k-instruct --wiki wikitext-2-raw-v1
!python /content/tinyllm-phase2/scripts/evaluate.py --which gptq --model_dir /content/tinyllm-phase2/models/phi3mini-gptq-4bit --wiki wikitext-2-raw-v1
!python /content/tinyllm-phase2/scripts/evaluate.py --which awq  --model_dir /content/tinyllm-phase2/models/phi3mini-awq-4bit  --wiki wikitext-2-raw-v1


## Export GGUF + run llama.cpp (CPU)

In [ ]:
%cd /content
!git clone -q https://github.com/ggerganov/llama.cpp
%cd llama.cpp
!make -j
%cd /content/tinyllm-phase2
!python scripts/export_to_gguf.py --hf_model microsoft/Phi-3-mini-4k-instruct --out_gguf models/phi3mini-f16.gguf
# Now run the printed command, then quantize and test main as shown.

## Plots

In [ ]:
!python /content/tinyllm-phase2/scripts/make_plots.py


## Notes
- If Phi-3 → GGUF conversion errors, try `mistralai/Mistral-7B-Instruct-v0.2`.
- Keep subsets small on CPU to finish quickly.
